# is-differentiable-flag composite — cx17: three-gate requires_grad: per-op flag short-circuits even when toggle is on

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `is-differentiable-flag`, `grad-tracking-global-toggle`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "is-differentiable-flag"
DD_ATOM_IDS = ["is-differentiable-flag", "grad-tracking-global-toggle"]
DD_SUBTOPICS = ["Backprop: is_differentiable flag", "Backprop: Grad-tracking toggle"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing the global grad-tracking toggle with the per-op is_differentiable flag

`requires_grad` on an output is the AND of three gates:

```
requires_grad = grad_tracking_enabled   # gate 1: global (runtime)
            AND is_differentiable        # gate 2: per-op (closure)
            AND any(input.requires_grad) # gate 3: inputs tracked
```

Gate 1 changes at RUNTIME (`set_grad_tracking(False)` from a `NoGrad` ctx). Gate 2 is captured ONCE at wrap-time and is sticky for the lifetime of that wrapper. The two are independent — turning the global toggle on does NOT make a non-differentiable op suddenly differentiable.

This composite has you implement both gates together and prove the independence with a state-table test.

### Composite Exercise — three-gate requires_grad: per-op flag short-circuits even when toggle is on

**Atoms exercised together**: `is-differentiable-flag`, `grad-tracking-global-toggle`

Implement THREE pieces:

**1. `set_grad_tracking(enabled)`** — write the module-level `grad_tracking_enabled` global (`globals()['grad_tracking_enabled'] = enabled`).

**2. `make_check_requires_grad(is_differentiable)`** — factory that returns a `check(args) -> bool` closing over `is_differentiable`. `check` reads `grad_tracking_enabled` from the module globals FRESH each call and ANDs all three gates.

**3. `cx17_wrap(fwd_fn, is_differentiable=True)`** — a tiny wrapper that uses the factory: returns a `tensor_func(*args)` that boxes the result in a `MiniTensor` and only builds a `Recipe` when the three-gate check returns True.

The test runs the full 2x2 truth table over `(toggle on/off, is_diff True/False)` to prove the gates are independent and BOTH must be True for a Recipe to appear.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

from dataclasses import dataclass, field
from typing import Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad=False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe

def set_grad_tracking(enabled: bool):
    raise NotImplementedError

def make_check_requires_grad(is_differentiable: bool):
    raise NotImplementedError

def cx17_wrap(fwd_fn, is_differentiable=True):
    """Wrapper that builds a Recipe only when all three gates pass."""
    raise NotImplementedError

def _test_cx17():
    # Reset the toggle in case a previous cell left it off.
    set_grad_tracking(True)
    assert globals()['grad_tracking_enabled'] is True

    add_diff    = cx17_wrap(t.add, is_differentiable=True)
    eq_nondiff  = cx17_wrap(t.eq, is_differentiable=False)

    a = MiniTensor(t.tensor([1.0, 2.0, 3.0]), requires_grad=True)
    b = MiniTensor(t.tensor([1.0, 0.0, 3.0]), requires_grad=True)

    # (i) toggle=True, is_diff=True, tracked inputs → requires_grad True, recipe attached
    out = add_diff(a, b)
    assert out.requires_grad is True
    assert out.recipe is not None
    assert out.recipe.func is t.add

    # (ii) toggle=True, is_diff=False, tracked inputs → False, NO recipe
    out = eq_nondiff(a, b)
    assert out.requires_grad is False, 'is_differentiable=False must short-circuit'
    assert out.recipe is None, 'non-diff op must NOT build Recipe even with toggle on'

    # (iii) toggle=False, is_diff=True, tracked inputs → False, NO recipe
    set_grad_tracking(False)
    out = add_diff(a, b)
    assert out.requires_grad is False, 'global toggle off must override is_differentiable=True'
    assert out.recipe is None

    # (iv) toggle=False, is_diff=False, tracked inputs → False (definitely)
    out = eq_nondiff(a, b)
    assert out.requires_grad is False and out.recipe is None

    # RESTORE toggle
    set_grad_tracking(True)

    # (v) per-op flag is STICKY across calls (closure capture works)
    # After many calls, eq_nondiff still says False — there is no way to flip it without re-wrapping.
    for _ in range(5):
        assert eq_nondiff(a, b).requires_grad is False

    # (vi) global toggle FLIPS add_diff back and forth at runtime (no re-wrap needed)
    set_grad_tracking(False)
    assert add_diff(a, b).requires_grad is False
    set_grad_tracking(True)
    assert add_diff(a, b).requires_grad is True

    # (vii) gate 3: untracked inputs always short-circuit
    u = MiniTensor(t.tensor([1.0]), requires_grad=False)
    v = MiniTensor(t.tensor([2.0]), requires_grad=False)
    out = add_diff(u, v)
    assert out.requires_grad is False and out.recipe is None

    # (viii) make_check_requires_grad doesn't cross-contaminate between factory calls
    c_diff = make_check_requires_grad(True)
    c_nondiff = make_check_requires_grad(False)
    assert c_diff((a,)) is True
    assert c_nondiff((a,)) is False
    assert c_diff((a,)) is True, 'second factory call leaked into first closure'
    _dd_passed.add('cx17')

_test_cx17()

<details><summary>Show solution — cx17</summary>

```python
def set_grad_tracking(enabled: bool):
    globals()['grad_tracking_enabled'] = enabled

def make_check_requires_grad(is_differentiable: bool):
    def check(args):
        return (
            globals()['grad_tracking_enabled']
            and is_differentiable
            and any(
                isinstance(a, MiniTensor) and a.requires_grad for a in args
            )
        )
    return check

def cx17_wrap(fwd_fn, is_differentiable=True):
    check = make_check_requires_grad(is_differentiable)
    def tensor_func(*args, **kwargs):
        raw_args = tuple(
            a.array if isinstance(a, MiniTensor) else a for a in args
        )
        out_raw = fwd_fn(*raw_args, **kwargs)
        rg = check(args)
        out = MiniTensor(out_raw, requires_grad=rg)
        if rg:
            parents = {
                i: a for i, a in enumerate(args) if isinstance(a, MiniTensor)
            }
            out.recipe = Recipe(fwd_fn, raw_args, kwargs, parents)
        return out
    return tensor_func
```

The independence point is the load-bearing one: turning the global toggle on does NOT re-enable a non-differentiable op. Gate 2 lives in the closure; you cannot flip it without re-wrapping.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx17'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx17',
        'subtopics': ["Backprop: is_differentiable flag", "Backprop: Grad-tracking toggle"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()